In [2]:
# ==============================================================================
# SCRIPT:         colab_asset_disposition.ipynb
# OPERATION:      Asset Disposition and Classification
# PURPOSE:        Analyzes the complete ground-truth asset ledger, classifies
#                 each asset as "Used", "Dropped (Duplicate)", or "Requires
#                 Investigation", and produces a final, auditable report.
# ==============================================================================

# --- Step 1: Install & Import ---
!pip install -q gspread
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default
from google.colab import drive
import os

# --- Step 2: Authentication & Mounting ---
print("Authenticating user for Google services...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ G-Sheets access authorized.")
print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully.")

# --- Step 3: Configuration ---
# A. Input file: The ground-truth asset data from Chrono-Forge.
INPUT_ASSET_CSV_PATH = '/content/drive/MyDrive/Opus_1_101001/Database/02_Cleaned_Data/chrono_forge_v6_output.csv'

# B. Input file: The live session log from Google Sheets.
SPREADSHEET_NAME = 'RAW_Requests'
WORKSHEET_NAME = 'raw_requests_polished'

# C. Output file: The destination for the final classification report.
OUTPUT_ANALYSIS_FOLDER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/'
OUTPUT_FILENAME = 'asset_disposition_report.csv'

# D. Column names from your data sources.
ASSET_FILENAME_COL = 'filename'
ASSET_HASH_COL = 'image_content_hash'
LOG_FILENAME_COL = 'Image'

# --- Step 4: MAIN EXECUTION ---
print("\n--- `Asset Disposition Analysis` ---")
try:
    # --- Phase 1: Load Datasets ---
    print("Loading datasets...")
    df_assets = pd.read_csv(INPUT_ASSET_CSV_PATH)
    worksheet = gc.open(SPREADSHEET_NAME).worksheet(WORKSHEET_NAME)
    df_log = pd.DataFrame(worksheet.get_all_records())
    print(f"Loaded {len(df_assets)} total assets and {len(df_log)} log entries.")

    # --- Phase 2: Establish Baselines ---
    print("Establishing baselines for classification...")
    # Get a fast lookup set of all filenames that are considered "Used".
    used_filenames = set(df_log[LOG_FILENAME_COL])

    # Get a fast lookup set of all image hashes from the "Used" files.
    used_assets_df = df_assets[df_assets[ASSET_FILENAME_COL].isin(used_filenames)]
    used_hashes = set(used_assets_df[ASSET_HASH_COL])
    print(f"Identified {len(used_filenames)} unique filenames and {len(used_hashes)} unique image hashes in the session log.")

    # --- Phase 3: Define and Apply Classification Logic ---
    def classify_asset(row):
        is_used = row[ASSET_FILENAME_COL] in used_filenames
        is_duplicate_hash = row[ASSET_HASH_COL] in used_hashes

        if is_used:
            return "Used"
        elif not is_used and is_duplicate_hash:
            return "Dropped (Duplicate)"
        else:
            return "Requires Investigation"

    print("\nClassifying all assets... (This may take a moment for thousands of files)")
    df_assets['Classification'] = df_assets.apply(classify_asset, axis=1)

    # --- Phase 4: Generate Reports and Save Output ---
    print("\n--- Classification Summary ---")
    classification_counts = df_assets['Classification'].value_counts()
    print(classification_counts.to_string())

    # Create the final, clean report with only the necessary columns
    report_df = df_assets[[
        'event_id',
        ASSET_HASH_COL,
        'containing_folder',
        ASSET_FILENAME_COL,
        'timestamp',
        'Classification'
    ]].sort_values(by=['containing_folder', ASSET_FILENAME_COL]) # Sort for readability

    # Save the full report
    full_output_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, OUTPUT_FILENAME)
    report_df.to_csv(full_output_path, index=False)
    print(f"\n✅ Full disposition report saved to your Google Drive at:")
    print(f"   > '{full_output_path}'")

    # Isolate and display the items that require attention
    investigation_df = report_df[report_df['Classification'] == 'Requires Investigation']
    if not investigation_df.empty:
        print("\n--- ⚠️ Action Required: Assets for Investigation ⚠️ ---")
        print(investigation_df.to_string(index=False))
    else:
        print("\n✅ No assets require further investigation.")

except Exception as e:
    print(f"\n❌ An unrecoverable error occurred: {e}")

Authenticating user for Google services...
✅ G-Sheets access authorized.

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully.

--- `Asset Disposition Analysis` ---
Loading datasets...
Loaded 5273 total assets and 4775 log entries.
Establishing baselines for classification...
Identified 4643 unique filenames and 4945 unique image hashes in the session log.

Classifying all assets... (This may take a moment for thousands of files)

--- Classification Summary ---
Classification
Used                      4984
Requires Investigation     288
Dropped (Duplicate)          1

✅ Full disposition report saved to your Google Drive at:
   > '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/asset_disposition_report.csv'

--- ⚠️ Action Required: Assets for Investigation ⚠️ ---
                                                        event_id              

In [6]:
# ==============================================================================
# SCRIPT:         Batched Manual Investigation Cell v2.1 (Corrected)
# OPERATION:      Grouped Interactive Triage
# PURPOSE:        Loads assets marked "Requires Investigation", sorts them by folder,
#                 then interactively prompts for manual classification in batches.
# ==============================================================================
import pandas as pd
import os

print("--- Initializing Batched Manual Investigation ---")

# --- Phase 1: Load and Prepare Data for Review ---
report_to_investigate_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, 'asset_disposition_report.csv')
final_manual_report_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, 'manual_disposition_complete.csv')

try:
    if not os.path.exists(report_to_investigate_path):
        raise FileNotFoundError("The asset disposition report was not found. Please run the previous cell first.")

    df_report = pd.read_csv(report_to_investigate_path)
    investigation_df = df_report[df_report['Classification'] == 'Requires Investigation'].copy()

    if investigation_df.empty:
        print("✅ No assets were flagged for investigation. Nothing to do.")
    else:
        # --- Sort the DataFrame before the loop ---
        investigation_df = investigation_df.sort_values(by=['containing_folder', 'timestamp']).reset_index(drop=True)
        print("Assets have been sorted by folder to streamline your workflow.")

        # --- Phase 2: Prepare for Interaction ---
        options_map = {
            '1': 'dropped, duplicate',
            '2': 'dropped, image not an offer',
            '3': 'dropped, other',
            '4': 'Requires further investigation at raw_requests'
        }

        print(f"\nFound {len(investigation_df)} assets requiring manual investigation.")
        print("Please classify each asset using the number codes:")
        print("-" * 50)
        for key, value in options_map.items(): print(f"  Enter '{key}' for: {value}")
        print("-" * 50)

        investigation_results = []
        total_items = len(investigation_df)
        last_folder_processed = None

        # --- Phase 3: The Interactive Classification Loop ---
        for index, row in investigation_df.iterrows():
            current_folder = row['containing_folder']

            # --- CORRECTED HEADER PRINTING BLOCK ---
            if current_folder != last_folder_processed:
                print("\n========================================================")
                print(f"--- NOW INVESTIGATING FOLDER: {current_folder} ---")
                print("========================================================")
                last_folder_processed = current_folder

            context_str = f"File: {row['filename']} | Time: {row['timestamp']}"
            print("\n" + context_str)

            while True:
                prompt = f"  -> Classify item [{len(investigation_results) + 1}/{total_items}]: "
                choice = input(prompt)
                if choice in options_map:
                    investigation_results.append(options_map[choice])
                    break
                else:
                    print(f"   ⚠️ Invalid choice. Please enter a number from 1 to 4.")

        # --- Phase 4: Finalize, Display, and Save ---
        print("\n--- Manual Investigation Complete ---")
        investigation_df['Investigation_Result'] = investigation_results

        print("\n--- Final Classification Report ---")
        final_columns = ['containing_folder', 'filename', 'Investigation_Result']
        print(investigation_df[final_columns].to_string())

        investigation_df.to_csv(final_manual_report_path, index=False)
        print(f"\n✅ Manually classified report saved to your Google Drive at:")
        print(f"   > '{final_manual_report_path}'")

except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}")
except Exception as e:
    print(f"\n❌ An unrecoverable error occurred: {e}")

--- Initializing Batched Manual Investigation ---
Assets have been sorted by folder to streamline your workflow.

Found 288 assets requiring manual investigation.
Please classify each asset using the number codes:
--------------------------------------------------
  Enter '1' for: dropped, duplicate
  Enter '2' for: dropped, image not an offer
  Enter '3' for: dropped, other
  Enter '4' for: Requires further investigation at raw_requests
--------------------------------------------------

--- NOW INVESTIGATING FOLDER: Screenshots_10 ---

File: IMG_6883.PNG | Time: 2025:08:27 08:49:30
  -> Classify item [1/288]: 2

--- NOW INVESTIGATING FOLDER: Screenshots_14 ---

File: IMG_7546.PNG | Time: 2025:08:29 06:26:04
  -> Classify item [2/288]: 2

File: IMG_7744.PNG | Time: 2025:08:29 08:39:51
  -> Classify item [3/288]: 2

--- NOW INVESTIGATING FOLDER: Screenshots_17 ---

File: IMG_7947.PNG | Time: 2025:08:31 16:12:04
  -> Classify item [4/288]: 2

--- NOW INVESTIGATING FOLDER: Screenshots_18

KeyboardInterrupt: Interrupted by user

In [7]:
# ==============================================================================
# SCRIPT:         Bulk Classification Cell (Executive Order)
# OPERATION:      Final Disposition
# PURPOSE:        Programmatically applies a default classification to all
#                 remaining un-investigated assets to finalize the report.
# ==============================================================================
import pandas as pd
import os

print("--- Executing Final Disposition Protocol ---")

# --- Phase 1: Load Data & Identify Targets ---
report_with_manual_entries_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, 'manual_disposition_complete.csv')

# The default status for all un-reviewed items.
DEFAULT_CLASSIFICATION = "Dropped (Assumed Non-Critical)"

try:
    if not os.path.exists(report_with_manual_entries_path):
        raise FileNotFoundError("The partially completed manual report was not found. Please run the interactive cell at least once.")

    # Load the results of the work you've already done.
    df_final = pd.read_csv(report_with_manual_entries_path)

    # --- Phase 2: Apply Bulk Classification ---
    # The .fillna() function is perfect for this. It finds all 'NaN' or empty values
    # in the 'Investigation_Result' column and fills them with our default status.
    print(f"Applying default status ('{DEFAULT_CLASSIFICATION}') to all remaining unclassified assets...")
    df_final['Investigation_Result'].fillna(DEFAULT_CLASSIFICATION, inplace=True)

    # --- Phase 3: Final Report Generation ---
    print("\n--- Final Bulk Classification Summary ---")
    print(df_final['Investigation_Result'].value_counts().to_string())

    final_report_path = os.path.join(OUTPUT_ANALYSIS_FOLDER_PATH, 'disposition_report_FINAL.csv')
    df_final.to_csv(final_report_path, index=False)

    print(f"\n✅ Operation Complete. The final, fully-classified disposition report has been saved to your Google Drive at:")
    print(f"   > '{final_report_path}'")

except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}")
except Exception as e:
    print(f"\n❌ An unrecoverable error occurred: {e}")

--- Executing Final Disposition Protocol ---

❌ ERROR: The partially completed manual report was not found. Please run the interactive cell at least once.
